# 96 — Build unified nodal stack manifest (SAFE, v5)

This notebook creates a deterministic manifest of the available nodal stack products.

The integration hierarchy is assigned from the **acquisition survey**, not merely
from the broad catalog branch. Survey labels are normalized, checked against explicit aliases, and then
classified using robust rules for `1m`, `2m`, and `streamer/MASW` labels:

| input product | priority | merge stage |
|---|---:|---|
| nodal stack linked to `T1_1m_refraction` | 1 | `reference` |
| nodal stack linked to `T1_2m_refraction` | 2 | `secondary_geode_extension` |
| nodal stack linked to `T1_streamer_masw` | 3 | `streamer_extension` |
| `nodal_only` stack | 4 | `candidate_extension` |

Lower priority numbers have greater authority when notebook 97 defines canonical
source clusters. This hierarchy is metadata only: notebook 96 does not cluster,
correlate, merge, or modify waveform products.

## Main outputs

- `96_nodal_stack_manifest.csv`
- `96_nodal_stack_file_manifest.csv`
- `96_nodal_stack_member_manifest.csv`
- `96_nodal_stack_receiver_manifest.csv`
- `96_nodal_stack_error_manifest.csv`
- `96_nodal_stack_manifest_summary.csv`


## 1. Configuration

In [1]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')

CATALOG_DB = PROJECT_ROOT / 'catalog' / 'lbssp_shot_catalog.sqlite'
NODAL_ONLY_EXPORT_ROOT = (
    PROJECT_ROOT
    / 'nodal_only_stacked_shot_gathers'
    / 'catalog_exports'
)

OUT_ROOT = PROJECT_ROOT / '96_unified_nodal_stack_manifest'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

REQUIRE_BOTH_BRANCHES = True
TARGET_LINES = None  # Example: ['T1']; None keeps all lines.

# Integration hierarchy. Survey labels are normalized before matching.
# Add future surveys or aliases here rather than changing notebook 97.
GEODE_SURVEY_POLICY = {
    't1_1m_refraction': {
        'priority': 1,
        'merge_stage': 'reference',
        'canonical_source_authority': True,
    },
    't1_2m_refraction': {
        'priority': 2,
        'merge_stage': 'secondary_geode_extension',
        'canonical_source_authority': False,
    },
    't1_streamer_masw': {
        'priority': 3,
        'merge_stage': 'streamer_extension',
        'canonical_source_authority': False,
    },
}

# Normalized aliases observed or plausibly used by upstream notebooks.
GEODE_SURVEY_ALIASES = {
    't1_1m': 't1_1m_refraction',
    't1_1m_refraction': 't1_1m_refraction',
    't1_1m_refraction_survey': 't1_1m_refraction',
    't1_2m': 't1_2m_refraction',
    't1_2m_refraction': 't1_2m_refraction',
    't1_2m_refraction_survey': 't1_2m_refraction',
    't1_streamer': 't1_streamer_masw',
    't1_streamer_masw': 't1_streamer_masw',
    'streamer_masw': 't1_streamer_masw',
    't1_masw_streamer': 't1_streamer_masw',
}

BRANCH_POLICY = {
    'nodal_only': {
        'priority': 4,
        'merge_stage': 'candidate_extension',
        'canonical_source_authority': False,
    },
}

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)

print('SQLite:', CATALOG_DB)
print('Nodal-only exports:', NODAL_ONLY_EXPORT_ROOT)
print('Output:', OUT_ROOT)
print('SAFE: no upstream database or waveform files are modified.')

SQLite: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
Nodal-only exports: /Volumes/tachyon/LBSSP_DATA/nodal_only_stacked_shot_gathers/catalog_exports
Output: /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_manifest
SAFE: no upstream database or waveform files are modified.


## 2. Helpers

In [2]:
def first_present(frame, candidates, default=np.nan):
    for name in candidates:
        if name in frame.columns:
            return frame[name]
    return pd.Series(default, index=frame.index)


def to_numeric_columns(frame, names):
    out = frame.copy()
    for name in names:
        if name in out.columns:
            out[name] = pd.to_numeric(out[name], errors='coerce')
    return out


def normalize_component(value):
    text = str(value).strip().upper()
    if text.endswith('Z'):
        return 'Z'
    if text.endswith('N'):
        return 'N'
    if text.endswith('E'):
        return 'E'
    return text


def parse_bool(series, default=False):
    return (
        series.fillna(default)
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(['true', '1', 'yes', 'y'])
    )


def path_exists(value):
    if pd.isna(value):
        return False
    text = str(value).strip()
    return bool(text) and Path(text).exists()


def station_code_to_x_m(station):
    try:
        return int(str(station)) / 100.0
    except Exception:
        return np.nan


def normalize_policy_label(value):
    if pd.isna(value):
        return None

    text = str(value).strip().lower()

    for old, new in [
        ('-', '_'),
        (' ', '_'),
        ('/', '_'),
        ('\\', '_'),
        ('.', '_'),
    ]:
        text = text.replace(old, new)

    while '__' in text:
        text = text.replace('__', '_')

    return text.strip('_')


def classify_geode_survey(survey):
    """Return canonical survey key plus a transparent classification reason."""
    normalized = normalize_policy_label(survey)

    if normalized is None:
        return None, 'missing_survey_label'

    # Explicit aliases take precedence.
    if normalized in GEODE_SURVEY_ALIASES:
        return GEODE_SURVEY_ALIASES[normalized], 'explicit_alias'

    tokens = set(normalized.split('_'))

    # Streamer/MASW must be checked first because labels may also contain
    # nominal spacing information.
    if (
        'streamer' in tokens
        or 'masw' in tokens
        or 'streamer' in normalized
        or 'masw' in normalized
    ):
        return 't1_streamer_masw', 'rule_streamer_or_masw'

    # Match common forms such as 1m, 1_m, 1meter, 1metre.
    one_metre_markers = [
        '1m', '1_m', '1meter', '1meters', '1metre', '1metres',
    ]
    two_metre_markers = [
        '2m', '2_m', '2meter', '2meters', '2metre', '2metres',
    ]

    has_one_metre = any(marker in normalized for marker in one_metre_markers)
    has_two_metre = any(marker in normalized for marker in two_metre_markers)

    if has_one_metre and not has_two_metre:
        return 't1_1m_refraction', 'rule_1m'
    if has_two_metre and not has_one_metre:
        return 't1_2m_refraction', 'rule_2m'

    return normalized, 'unmatched'


def resolve_geode_survey_policy(survey):
    canonical_key, reason = classify_geode_survey(survey)
    return canonical_key, GEODE_SURVEY_POLICY.get(canonical_key), reason

def apply_integration_policy(frame):
    out = frame.copy()

    priority = []
    merge_stage = []
    canonical_authority = []
    policy_key = []
    normalized_survey = []
    survey_classification_reason = []

    for row in out.itertuples(index=False):
        branch = str(row.catalog_branch)

        if branch == 'geode_linked':
            canonical_survey_key, policy, reason = resolve_geode_survey_policy(
                row.survey
            )
            key = f'geode_linked:{canonical_survey_key}'
            normalized_survey.append(canonical_survey_key)
            survey_classification_reason.append(reason)
        else:
            canonical_survey_key = normalize_policy_label(row.survey)
            policy = BRANCH_POLICY.get(branch)
            key = branch
            normalized_survey.append(canonical_survey_key)
            survey_classification_reason.append('branch_policy')

        if policy is None:
            priority.append(np.nan)
            merge_stage.append('unclassified')
            canonical_authority.append(False)
            policy_key.append(key)
            continue

        priority.append(policy['priority'])
        merge_stage.append(policy['merge_stage'])
        canonical_authority.append(
            policy.get('canonical_source_authority', False)
        )
        policy_key.append(key)

    out['survey_normalized'] = normalized_survey
    out['survey_classification_reason'] = survey_classification_reason
    out['priority'] = priority
    out['merge_stage'] = merge_stage
    out['canonical_source_authority'] = canonical_authority
    out['integration_policy_key'] = policy_key
    return out


## 3. Load notebook-95 SQLite products

In [3]:
GEODE_TABLES = {
    'stacks': 'nodal_stacks',
    'members': 'nodal_stack_members',
    'files': 'nodal_stack_files',
    'errors': 'nodal_stack_processing_errors',
}

if not CATALOG_DB.exists():
    if REQUIRE_BOTH_BRANCHES:
        raise FileNotFoundError(f'Missing SQLite database: {CATALOG_DB}')
    geode_raw = {name: pd.DataFrame() for name in GEODE_TABLES}
else:
    uri = f'file:{CATALOG_DB}?mode=ro'
    with sqlite3.connect(uri, uri=True) as connection:
        available = set(pd.read_sql(
            "SELECT name FROM sqlite_master WHERE type='table'",
            connection,
        )['name'])

        missing = sorted(set(GEODE_TABLES.values()) - available)
        if missing:
            raise RuntimeError(f'Missing notebook-95 SQLite tables: {missing}')

        geode_raw = {
            key: pd.read_sql(f'SELECT * FROM {table}', connection)
            for key, table in GEODE_TABLES.items()
        }

for key, frame in geode_raw.items():
    print(f'Geode-linked {key}: {len(frame)}')

Geode-linked stacks: 194
Geode-linked members: 961
Geode-linked files: 582
Geode-linked errors: 0


## 4. Load notebook-95c CSV products

In [4]:
NODAL_ONLY_FILES = {
    'stacks': NODAL_ONLY_EXPORT_ROOT / 'nodal_only_stacks.csv',
    'members': NODAL_ONLY_EXPORT_ROOT / 'nodal_only_stack_members.csv',
    'files': NODAL_ONLY_EXPORT_ROOT / 'nodal_only_stack_files.csv',
    'receivers': NODAL_ONLY_EXPORT_ROOT / 'nodal_only_stack_trace_contributions.csv',
    'errors': NODAL_ONLY_EXPORT_ROOT / 'nodal_only_stack_processing_errors.csv',
}

missing = [key for key, path in NODAL_ONLY_FILES.items() if not path.exists()]
if missing and REQUIRE_BOTH_BRANCHES:
    lines = '\n'.join(f'  {key}: {NODAL_ONLY_FILES[key]}' for key in missing)
    raise FileNotFoundError(
        'Missing notebook-95c exports. Run notebook 95c first.\n' + lines
    )


def read_optional_csv(path):
    return pd.read_csv(path, low_memory=False) if path.exists() else pd.DataFrame()


nodal_only_raw = {
    key: read_optional_csv(path)
    for key, path in NODAL_ONLY_FILES.items()
}

for key, frame in nodal_only_raw.items():
    print(f'Nodal-only {key}: {len(frame)}')

Nodal-only stacks: 44
Nodal-only members: 832
Nodal-only files: 396
Nodal-only receivers: 4464
Nodal-only errors: 0


## 5. Standardize stack manifest

In [5]:
def standardize_geode_stacks(frame):
    if frame.empty:
        return pd.DataFrame()

    frame = to_numeric_columns(frame, [
        'file_no', 'source_x_truth_m', 'n_candidate_members',
        'n_accepted_members', 'n_rejected_members',
        'median_xcorr_shift_s', 'median_xcorr_corrcoef',
    ])

    out = pd.DataFrame(index=frame.index)
    out['stack_id'] = frame['stack_id'].astype(str)
    out['catalog_branch'] = 'geode_linked'
    out['stack_basis'] = 'nodal_events_linked_to_geode_stack'
    out['line'] = first_present(frame, ['line'])
    out['survey'] = first_present(frame, ['geode_survey', 'survey'])
    out['source_x_m'] = first_present(frame, ['source_x_truth_m'])
    out['source_position_authority'] = 'surveyed_geode_source'
    out['source_position_status'] = 'truth_position_from_geode_metadata'
    out['source_type'] = first_present(frame, ['source_type'])
    out['geode_event_id'] = first_present(frame, ['geode_event_id'])
    out['geode_file_no'] = first_present(frame, ['file_no'])
    out['nodal_only_group_key'] = None
    out['reference_nodal_event_id'] = first_present(
        frame, ['reference_nodal_event_id']
    )
    out['n_candidate_members'] = first_present(frame, ['n_candidate_members'])
    out['n_accepted_members'] = first_present(frame, ['n_accepted_members'])
    out['n_rejected_members'] = first_present(frame, ['n_rejected_members'])
    out['median_xcorr_shift_s'] = first_present(frame, ['median_xcorr_shift_s'])
    out['median_xcorr_corrcoef'] = first_present(
        frame, ['median_xcorr_corrcoef']
    )
    out['minimum_xcorr_corrcoef'] = np.nan
    out['expected_blows'] = np.nan
    out['components_written'] = np.nan
    out['output_directory'] = first_present(
        frame, ['output_dir', 'output_directory']
    )
    out['status'] = first_present(frame, ['status'], default='ok')
    return apply_integration_policy(out)


def standardize_nodal_only_stacks(frame):
    if frame.empty:
        return pd.DataFrame()

    frame = to_numeric_columns(frame, [
        'assigned_source_x_m', 'n_candidate_members',
        'n_accepted_members', 'n_rejected_members', 'expected_blows',
        'median_xcorr_shift_s', 'median_xcorr_corrcoef',
        'minimum_xcorr_corrcoef',
    ])

    out = pd.DataFrame(index=frame.index)
    out['stack_id'] = frame['stack_id'].astype(str)
    out['catalog_branch'] = 'nodal_only'
    out['stack_basis'] = first_present(
        frame, ['stack_basis'], default='nodal_only_inferred_group'
    )
    out['line'] = first_present(frame, ['line'], default='T1')
    out['survey'] = first_present(frame, ['survey', 'nodal_timewindow_label'])
    out['source_x_m'] = first_present(frame, ['assigned_source_x_m'])
    out['source_position_authority'] = 'assigned_nodal_only_source'
    out['source_position_status'] = first_present(
        frame, ['source_position_status'],
        default='assigned_in_94b_reviewed_in_94c',
    )
    out['source_type'] = first_present(frame, ['source_type'])
    out['geode_event_id'] = None
    out['geode_file_no'] = np.nan
    out['nodal_only_group_key'] = first_present(frame, ['group_key'])
    out['reference_nodal_event_id'] = first_present(
        frame, ['reference_nodal_event_id']
    )
    out['n_candidate_members'] = first_present(
        frame, ['n_candidate_members', 'n_accepted_members']
    )
    out['n_accepted_members'] = first_present(frame, ['n_accepted_members'])
    out['n_rejected_members'] = first_present(
        frame, ['n_rejected_members'], default=0
    )
    out['median_xcorr_shift_s'] = first_present(frame, ['median_xcorr_shift_s'])
    out['median_xcorr_corrcoef'] = first_present(
        frame, ['median_xcorr_corrcoef']
    )
    out['minimum_xcorr_corrcoef'] = first_present(
        frame, ['minimum_xcorr_corrcoef']
    )
    out['expected_blows'] = first_present(frame, ['expected_blows'])
    out['components_written'] = first_present(frame, ['components_written'])
    out['output_directory'] = first_present(
        frame, ['output_directory', 'output_dir']
    )
    out['status'] = first_present(frame, ['status'], default='stack_created')
    return apply_integration_policy(out)


stack_manifest = pd.concat(
    [
        standardize_geode_stacks(geode_raw['stacks']),
        standardize_nodal_only_stacks(nodal_only_raw['stacks']),
    ],
    ignore_index=True,
    sort=False,
)

stack_manifest['source_x_m'] = pd.to_numeric(
    stack_manifest['source_x_m'], errors='coerce'
)
stack_manifest['priority'] = pd.to_numeric(
    stack_manifest['priority'], errors='coerce'
).astype('Int64')

if TARGET_LINES is not None:
    stack_manifest = stack_manifest.loc[
        stack_manifest.line.astype(str).isin(TARGET_LINES)
    ].copy()

duplicates = stack_manifest.loc[
    stack_manifest.stack_id.duplicated(keep=False),
    ['stack_id', 'catalog_branch'],
]
if len(duplicates):
    display(duplicates)
    raise RuntimeError('stack_id must be unique across all branches.')

unknown_policy = stack_manifest.loc[
    stack_manifest.priority.isna(),
    [
        'stack_id',
        'catalog_branch',
        'survey',
        'survey_normalized',
        'survey_classification_reason',
        'integration_policy_key',
    ],
]
if len(unknown_policy):
    print('Unmatched survey labels:')
    display(
        unknown_policy.groupby(
            [
                'catalog_branch',
                'survey',
                'survey_normalized',
                'survey_classification_reason',
                'integration_policy_key',
            ],
            dropna=False,
        ).size().reset_index(name='n_stacks')
    )
    raise RuntimeError(
        'Some stacks did not match the integration policy. '
        'Add the displayed normalized survey label to '
        'GEODE_SURVEY_ALIASES or GEODE_SURVEY_POLICY.'
    )

stack_manifest = stack_manifest.sort_values(
    ['line', 'source_x_m', 'priority', 'stack_id'],
    na_position='last',
).reset_index(drop=True)

print('Unified stack rows:', len(stack_manifest))
display(
    stack_manifest.groupby(
        [
            'catalog_branch', 'survey', 'survey_normalized', 'survey_classification_reason', 'integration_policy_key',
            'priority', 'merge_stage',
        ],
        dropna=False,
    ).size().reset_index(name='n_stacks')
)
display(stack_manifest.head(20))

Unified stack rows: 238


,catalog_branch,survey,survey_normalized,survey_classification_reason,integration_policy_key,priority,merge_stage,n_stacks
0,geode_linked,T1_1m_refraction,t1_1m_refraction,explicit_alias,geode_linked:t1_1m_refraction,1,reference,39
1,geode_linked,T1_2m_refraction,t1_2m_refraction,explicit_alias,geode_linked:t1_2m_refraction,2,secondary_geode_extension,36
2,geode_linked,T1_streamer_masw,t1_streamer_masw,explicit_alias,geode_linked:t1_streamer_masw,3,streamer_extension,80
3,geode_linked,T3_1m_refraction,t1_1m_refraction,rule_1m,geode_linked:t1_1m_refraction,1,reference,39
4,nodal_only,NaN,NaN,branch_policy,nodal_only,4,candidate_extension,44


,stack_id,catalog_branch,stack_basis,line,survey,source_x_m,source_position_authority,source_position_status,source_type,geode_event_id,geode_file_no,nodal_only_group_key,reference_nodal_event_id,n_candidate_members,n_accepted_members,n_rejected_members,median_xcorr_shift_s,median_xcorr_corrcoef,minimum_xcorr_corrcoef,expected_blows,components_written,output_directory,status,survey_normalized,survey_classification_reason,priority,merge_stage,canonical_source_authority,integration_policy_key
0,NODALONLYSTACK_MAY19_010M_x0010.0m,nodal_only,nodal_only_inferred_group,T1,NaN,10.0,assigned_nodal_only_source,assigned_in_94b_reviewed_in_94c,NaN,None,NaN,MAY19_010M,T1_N3_Nodal3_T1_N3_E00007,43,43,0,-0.004,0.968000,0.8652,60.0,"Z,N,E",/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,stack_created,None,branch_policy,4,candidate_extension,False,nodal_only
1,NODALONLYSTACK_MAY19_036M_x0036.0m,nodal_only,nodal_only_inferred_group,T1,NaN,36.0,assigned_nodal_only_source,assigned_in_94b_reviewed_in_94c,NaN,None,NaN,MAY19_036M,RECOV_MAY19_036M_0005,22,22,0,-0.002,0.978900,0.9356,22.0,"Z,N,E",/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,stack_created,None,branch_policy,4,candidate_extension,False,nodal_only
2,NODALSTACK_T1_T1_2m_refraction_F3047_x0043.0m,geode_linked,nodal_events_linked_to_geode_stack,T1,T1_2m_refraction,43.0,surveyed_geode_source,truth_position_from_geode_metadata,hammer,GEODE_T1_2M_REFRACTION_F3047,3047.0,None,T1_N2_Refraction2m_T1_N2_E00009,6,6,0,0.018,0.975976,NaN,NaN,NaN,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok,t1_2m_refraction,explicit_alias,2,secondary_geode_extension,False,geode_linked:t1_2m_refraction
3,NODALSTACK_T1_T1_2m_refraction_F3048_x0047.0m,geode_linked,nodal_events_linked_to_geode_stack,T1,T1_2m_refraction,47.0,surveyed_geode_source,truth_position_from_geode_metadata,hammer,GEODE_T1_2M_REFRACTION_F3048,3048.0,None,T1_N2_Refraction2m_T1_N2_E00016,7,7,0,-0.022,0.986214,NaN,NaN,NaN,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok,t1_2m_refraction,explicit_alias,2,secondary_geode_extension,False,geode_linked:t1_2m_refraction
4,NODALSTACK_T1_T1_2m_refraction_F3049_x0051.0m,geode_linked,nodal_events_linked_to_geode_stack,T1,T1_2m_refraction,51.0,surveyed_geode_source,truth_position_from_geode_metadata,hammer,GEODE_T1_2M_REFRACTION_F3049,3049.0,None,T1_N2_Refraction2m_T1_N2_E00027,6,6,0,0.000,0.984600,NaN,NaN,NaN,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok,t1_2m_refraction,explicit_alias,2,secondary_geode_extension,False,geode_linked:t1_2m_refraction
5,NODALSTACK_T1_T1_2m_refraction_F3050_x0055.0m,geode_linked,nodal_events_linked_to_geode_stack,T1,T1_2m_refraction,55.0,surveyed_geode_source,truth_position_from_geode_metadata,hammer,GEODE_T1_2M_REFRACTION_F3050,3050.0,None,T1_N2_Refraction2m_T1_N2_E00039,6,6,0,0.008,0.979924,NaN,NaN,NaN,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok,t1_2m_refraction,explicit_alias,2,secondary_geode_extension,False,geode_linked:t1_2m_refraction
6,NODALSTACK_T1_T1_2m_refraction_F3051_x0059.0m,geode_linked,nodal_events_linked_to_geode_stack,T1,T1_2m_refraction,59.0,surveyed_geode_source,truth_position_from_geode_metadata,hammer,GEODE_T1_2M_REFRACTION_F3051,3051.0,None,T1_N2_Refraction2m_T1_N2_E00046,6,6,0,0.000,0.986099,NaN,NaN,NaN,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok,t1_2m_refraction,explicit_alias,2,secondary_geode_extension,False,geode_linked:t1_2m_refraction
7,NODALSTACK_T1_T1_2m_refraction_F3052_x0063.0m,geode_linked,nodal_events_linked_to_geode_stack,T1,T1_2m_refraction,63.0,surveyed_geode_source,truth_position_from_geode_metadata,hammer,GEODE_T1_2M_REFRACTION_F3052,3052.0,None,T1_N2_Refraction2m_T1_N2_E00060,7,7,0,0.078,0.986585,NaN,NaN,NaN,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok,t1_2m_refraction,explicit_alias,2,secondary_geode_extension,False,geode_linked:t1_2m_refraction
8,NODALSTACK_T1_T1_2m_refraction_F3053_x0067.0m,geode_linked,nodal_events_linked_to_geode_stack,T1,T1_2m_refraction,67.0,surveyed_geode_source,tr

## 6. Standardize file manifest and validate indexed paths

In [6]:
def standardize_files(frame, branch):
    if frame.empty:
        return pd.DataFrame(columns=[
            'stack_id', 'catalog_branch', 'component', 'file_type',
            'file_path', 'n_traces', 'n_stack_members',
        ])

    out = pd.DataFrame(index=frame.index)
    out['stack_id'] = frame['stack_id'].astype(str)
    out['catalog_branch'] = branch
    out['component'] = first_present(frame, ['component']).map(
        normalize_component
    )
    out['file_type'] = first_present(frame, ['file_type']).astype(str)
    out['file_path'] = first_present(frame, ['file_path']).astype(str)
    out['n_traces'] = pd.to_numeric(
        first_present(frame, ['n_traces']), errors='coerce'
    )
    out['n_stack_members'] = pd.to_numeric(
        first_present(frame, ['n_stack_members']), errors='coerce'
    )
    return out


file_manifest = pd.concat(
    [
        standardize_files(geode_raw['files'], 'geode_linked'),
        standardize_files(nodal_only_raw['files'], 'nodal_only'),
    ],
    ignore_index=True,
    sort=False,
)

file_manifest = file_manifest.merge(
    stack_manifest[
        [
            'stack_id', 'line', 'source_x_m', 'source_position_authority',
            'priority', 'merge_stage',
        ]
    ],
    on='stack_id',
    how='inner',
    validate='many_to_one',
)

file_manifest['file_exists'] = file_manifest.file_path.map(path_exists)
file_manifest['file_size_bytes'] = file_manifest.file_path.map(
    lambda value: Path(str(value)).stat().st_size
    if path_exists(value) else np.nan
)

file_manifest = file_manifest.sort_values(
    ['line', 'source_x_m', 'priority', 'stack_id', 'component', 'file_type']
).reset_index(drop=True)

print('Indexed products:', len(file_manifest))
print('Missing indexed products:', int((~file_manifest.file_exists).sum()))
display(
    file_manifest.groupby(
        ['catalog_branch', 'component', 'file_type', 'file_exists'],
        dropna=False,
    ).size().reset_index(name='n_files')
)

Indexed products: 978
Missing indexed products: 0


,catalog_branch,component,file_type,file_exists,n_files
0,geode_linked,Z,mseed,True,194
1,geode_linked,Z,png_wiggle,True,194
2,geode_linked,Z,segy,True,194
3,nodal_only,E,mseed,True,44
4,nodal_only,E,png_wiggle,True,44
5,nodal_only,E,segy,True,44
6,nodal_only,N,mseed,True,44
7,nodal_only,N,png_wiggle,True,44
8,nodal_only,N,segy,True,44
9,nodal_only,Z,mseed,True,44


## 7. Standardize member manifest

In [7]:
def standardize_members(frame, branch):
    if frame.empty:
        return pd.DataFrame()

    out = pd.DataFrame(index=frame.index)
    out['stack_id'] = frame['stack_id'].astype(str)
    out['catalog_branch'] = branch
    out['nodal_event_id'] = first_present(frame, ['nodal_event_id'])
    out['event_time_utc'] = first_present(
        frame, ['nodal_event_time_utc', 'event_time_utc', 'event_time']
    )
    out['corrected_event_time_utc'] = first_present(
        frame, ['corrected_nodal_event_time_utc', 'corrected_event_time_utc']
    )
    out['xcorr_shift_s'] = pd.to_numeric(
        first_present(frame, ['xcorr_shift_s']), errors='coerce'
    )
    out['xcorr_corrcoef'] = pd.to_numeric(
        first_present(frame, ['xcorr_corrcoef']), errors='coerce'
    )
    out['xcorr_n_traces'] = pd.to_numeric(
        first_present(frame, ['xcorr_n_traces']), errors='coerce'
    )
    out['xcorr_shift_mad_s'] = pd.to_numeric(
        first_present(frame, ['xcorr_shift_mad_s']), errors='coerce'
    )
    out['is_reference_event'] = parse_bool(
        first_present(frame, ['is_reference_event'])
    )
    out['included_in_stack'] = parse_bool(
        first_present(
            frame,
            [
                'included_in_output_stack', 'included_in_stack',
                'accepted_for_stack',
            ],
            default=True,
        )
    )
    out['waveform_qc_status'] = first_present(frame, ['waveform_qc_status'])
    out['source_member_path'] = first_present(
        frame, ['long_mseed_path', 'mseed_path', 'resolved_gather_path']
    )
    return out


member_manifest = pd.concat(
    [
        standardize_members(geode_raw['members'], 'geode_linked'),
        standardize_members(nodal_only_raw['members'], 'nodal_only'),
    ],
    ignore_index=True,
    sort=False,
)

if len(member_manifest):
    member_manifest = member_manifest.merge(
        stack_manifest[
            ['stack_id', 'line', 'source_x_m', 'priority', 'merge_stage']
        ],
        on='stack_id',
        how='inner',
        validate='many_to_one',
    )

print('Member rows:', len(member_manifest))
if len(member_manifest):
    display(
        member_manifest.groupby(
            ['catalog_branch', 'included_in_stack'],
            dropna=False,
        ).size().reset_index(name='n_members')
    )

Member rows: 1793


,catalog_branch,included_in_stack,n_members
0,geode_linked,False,6
1,geode_linked,True,955
2,nodal_only,True,832


## 8. Build receiver manifest

In [8]:
try:
    from obspy import read
except Exception as exc:
    read = None
    print('WARNING: ObsPy could not be imported:', repr(exc))


receiver_rows = []

# Notebook 95c provides a direct trace-contribution catalog.
for row in nodal_only_raw['receivers'].itertuples(index=False):
    receiver_rows.append({
        'stack_id': str(row.stack_id),
        'catalog_branch': 'nodal_only',
        'component': normalize_component(getattr(row, 'component', '')),
        'station': str(getattr(row, 'station', '')),
        'channel': str(getattr(row, 'channel', '')),
        'receiver_x_m': pd.to_numeric(
            getattr(row, 'receiver_x_m', np.nan), errors='coerce'
        ),
        'n_contributing_events': pd.to_numeric(
            getattr(row, 'n_contributing_events', np.nan), errors='coerce'
        ),
        'receiver_position_source': '95c_trace_contribution_catalog',
        'source_file_path': None,
    })

# For notebook 95, inspect indexed MiniSEED products.
geode_mseed = file_manifest.loc[
    file_manifest.catalog_branch.eq('geode_linked')
    & file_manifest.file_type.str.lower().eq('mseed')
    & file_manifest.file_exists
].copy()

if read is not None:
    for row in geode_mseed.itertuples(index=False):
        try:
            stream = read(str(row.file_path))
            for trace in stream:
                receiver_x = pd.to_numeric(
                    getattr(trace.stats, 'receiver_x_m', np.nan),
                    errors='coerce',
                )
                if pd.isna(receiver_x):
                    receiver_x = station_code_to_x_m(trace.stats.station)

                receiver_rows.append({
                    'stack_id': str(row.stack_id),
                    'catalog_branch': 'geode_linked',
                    'component': normalize_component(trace.stats.channel),
                    'station': str(trace.stats.station),
                    'channel': str(trace.stats.channel),
                    'receiver_x_m': receiver_x,
                    'n_contributing_events': row.n_stack_members,
                    'receiver_position_source': (
                        '95_mseed_stats_or_position_coded_station'
                    ),
                    'source_file_path': str(row.file_path),
                })
        except Exception as exc:
            print(
                'WARNING: receiver inspection failed:',
                row.file_path,
                repr(exc),
            )

receiver_manifest = pd.DataFrame(receiver_rows)

if len(receiver_manifest):
    receiver_manifest['receiver_x_m'] = pd.to_numeric(
        receiver_manifest.receiver_x_m, errors='coerce'
    )
    receiver_manifest = receiver_manifest.merge(
        stack_manifest[
            [
                'stack_id', 'line', 'source_x_m', 'priority', 'merge_stage',
            ]
        ],
        on='stack_id',
        how='inner',
        validate='many_to_one',
    )
    receiver_manifest = receiver_manifest.sort_values(
        [
            'line', 'source_x_m', 'priority', 'stack_id',
            'component', 'receiver_x_m',
        ]
    ).reset_index(drop=True)

print('Receiver rows:', len(receiver_manifest))
if len(receiver_manifest):
    display(
        receiver_manifest.groupby(
            ['catalog_branch', 'component'],
            dropna=False,
        ).agg(
            n_records=('receiver_x_m', 'size'),
            n_positions=('receiver_x_m', 'nunique'),
            receiver_x_min_m=('receiver_x_m', 'min'),
            receiver_x_max_m=('receiver_x_m', 'max'),
        ).reset_index()
    )

Receiver rows: 11179


,catalog_branch,component,n_records,n_positions,receiver_x_min_m,receiver_x_max_m
0,geode_linked,Z,6715,87,0.5,216.0
1,nodal_only,E,1488,51,28.0,216.0
2,nodal_only,N,1488,51,28.0,216.0
3,nodal_only,Z,1488,51,28.0,216.0


## 9. Standardize processing-error manifest

In [9]:
def standardize_errors(frame, branch):
    if frame.empty:
        return pd.DataFrame(columns=[
            'catalog_branch', 'stack_id', 'group_or_event_id',
            'stage', 'error', 'traceback',
        ])

    out = pd.DataFrame(index=frame.index)
    out['catalog_branch'] = branch
    out['stack_id'] = first_present(frame, ['stack_id'])
    out['group_or_event_id'] = first_present(
        frame, ['group_key', 'geode_event_id']
    )
    out['stage'] = first_present(frame, ['stage'])
    out['error'] = first_present(frame, ['error'])
    out['traceback'] = first_present(frame, ['traceback'])
    return out


error_manifest = pd.concat(
    [
        standardize_errors(geode_raw['errors'], 'geode_linked'),
        standardize_errors(nodal_only_raw['errors'], 'nodal_only'),
    ],
    ignore_index=True,
    sort=False,
)

print('Processing-error rows:', len(error_manifest))
if len(error_manifest):
    display(
        error_manifest.groupby(
            ['catalog_branch', 'stage'], dropna=False
        ).size().reset_index(name='n_errors')
    )

Processing-error rows: 0


## 10. Integrity checks

In [10]:
issues = []

if stack_manifest.empty:
    issues.append('No stack rows were loaded.')

missing_positions = int(stack_manifest.source_x_m.isna().sum())
if missing_positions:
    issues.append(f'{missing_positions} stack rows have no source_x_m.')

missing_files = file_manifest.loc[~file_manifest.file_exists].copy()
if len(missing_files):
    issues.append(f'{len(missing_files)} indexed products do not exist.')

existing_mseed_ids = set(
    file_manifest.loc[
        file_manifest.file_type.str.lower().eq('mseed')
        & file_manifest.file_exists,
        'stack_id',
    ]
)
stacks_without_mseed = sorted(set(stack_manifest.stack_id) - existing_mseed_ids)
if stacks_without_mseed:
    issues.append(
        f'{len(stacks_without_mseed)} stacks have no existing indexed MiniSEED product.'
    )

if stack_manifest.priority.isna().any():
    issues.append('At least one stack has no integration priority.')

print('Integrity issues:', len(issues))
for item in issues:
    print(' -', item)

if len(missing_files):
    display(missing_files[
        ['stack_id', 'catalog_branch', 'component', 'file_type', 'file_path']
    ].head(50))

if stacks_without_mseed:
    display(stack_manifest.loc[
        stack_manifest.stack_id.isin(stacks_without_mseed),
        [
            'stack_id', 'catalog_branch', 'line',
            'source_x_m', 'priority', 'status',
        ],
    ])

if not issues:
    print('PASS: unified nodal stack manifest is internally consistent.')
else:
    print('REVIEW: outputs will be written, but inspect the issues above.')

Integrity issues: 0
PASS: unified nodal stack manifest is internally consistent.


## 11. Export manifests

In [11]:
OUTPUTS = {
    'stacks': OUT_ROOT / '96_nodal_stack_manifest.csv',
    'files': OUT_ROOT / '96_nodal_stack_file_manifest.csv',
    'members': OUT_ROOT / '96_nodal_stack_member_manifest.csv',
    'receivers': OUT_ROOT / '96_nodal_stack_receiver_manifest.csv',
    'errors': OUT_ROOT / '96_nodal_stack_error_manifest.csv',
    'summary': OUT_ROOT / '96_nodal_stack_manifest_summary.csv',
}

stack_manifest.to_csv(OUTPUTS['stacks'], index=False)
file_manifest.to_csv(OUTPUTS['files'], index=False)
member_manifest.to_csv(OUTPUTS['members'], index=False)
receiver_manifest.to_csv(OUTPUTS['receivers'], index=False)
error_manifest.to_csv(OUTPUTS['errors'], index=False)

summary_rows = [
    ('stack_rows', len(stack_manifest)),
    ('indexed_products', len(file_manifest)),
    ('missing_indexed_products', int((~file_manifest.file_exists).sum())),
    ('member_rows', len(member_manifest)),
    ('receiver_rows', len(receiver_manifest)),
    ('processing_error_rows', len(error_manifest)),
    ('integrity_issue_count', len(issues)),
]

for policy_key, policy_frame in stack_manifest.groupby(
    'integration_policy_key', dropna=False
):
    safe_key = str(policy_key).replace(':', '_')
    summary_rows.extend([
        (f'{safe_key}_stacks', len(policy_frame)),
        (f'{safe_key}_priority', int(policy_frame.priority.iloc[0])),
        (f'{safe_key}_merge_stage', policy_frame.merge_stage.iloc[0]),
    ])

summary = pd.DataFrame(summary_rows, columns=['metric', 'value'])
summary.to_csv(OUTPUTS['summary'], index=False)
display(summary)

print('\nWritten:')
for key, path in OUTPUTS.items():
    print(f'  {key:10s} {path}')

,metric,value
0,stack_rows,238
1,indexed_products,978
2,missing_indexed_products,0
3,member_rows,1793
4,receiver_rows,11179
5,processing_error_rows,0
6,integrity_issue_count,0
7,geode_linked_t1_1m_refraction_stacks,78
8,geode_linked_t1_1m_refraction_priority,1
9,geode_linked_t1_1m_refraction_merge_stage,reference



Written:
  stacks     /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_manifest/96_nodal_stack_manifest.csv
  files      /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_manifest/96_nodal_stack_file_manifest.csv
  members    /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_manifest/96_nodal_stack_member_manifest.csv
  receivers  /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_manifest/96_nodal_stack_receiver_manifest.csv
  errors     /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_manifest/96_nodal_stack_error_manifest.csv
  summary    /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_manifest/96_nodal_stack_manifest_summary.csv


## 12. Next step

Notebook 97 performs three distinct stages:

1. **Geometry:** assign stacks to canonical source clusters using the
   survey-derived priority and a 0.25 m source tolerance.
2. **Tasks:** generate all pairwise stack comparisons within multi-member clusters.
3. **Evidence:** match receivers and calculate waveform-correlation diagnostics.

Notebook 97 already reads the hierarchy from the exported manifests, so it does
not require a code change. Rerun notebook 97 after rerunning this notebook.

No waveform products are merged in notebook 97.
